In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from catboost import CatBoostClassifier, Pool
import joblib

df = pd.read_csv("atp_matches_feature_engineer.csv")

In [23]:
# 2. Convert boolean columns to integers (0 and 1)
# This ensures full compatibility with all XGBoost versions
bool_cols = df.select_dtypes(include=['bool']).columns
df[bool_cols] = df[bool_cols].astype(int)

# 3. Define Features (X) and Target (y)
X = df.drop(columns=['target'])
y = df['target']

# 4. Split into Training and Testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
tscv = TimeSeriesSplit(n_splits=5)


In [24]:
# 3. Build the Pipeline
pipe = Pipeline([
    ('xgb', xgb.XGBClassifier(
        random_state=42, 
        use_label_encoder=False, 
        eval_metric='logloss'
    ))
])

# 4. Define Hyperparameter Search Space
param_dist = {
    'xgb__n_estimators': [100, 300, 500],
    'xgb__learning_rate': [0.01, 0.05, 0.1],
    'xgb__max_depth': [3, 4, 6],
    'xgb__subsample': [0.7, 0.8, 0.9],
    'xgb__colsample_bytree': [0.7, 0.8, 0.9],
    'xgb__gamma': [0, 0.1, 0.2]
}


In [25]:
# 5. Run the Randomized Search
print("Starting Hyperparameter Tuning...")
search = RandomizedSearchCV(
    pipe,
    param_distributions=param_dist,
    n_iter=15,           # Increase to 50+ for better results if you have time
    cv=tscv,             # Uses our Time-Series Split
    scoring='roc_auc',   # Optimizing for probability accuracy
    n_jobs=-1,           # Uses all CPU cores
    verbose=1,
    random_state=42
)

search.fit(X, y)

# 6. Results and Evaluation
best_model = search.best_estimator_
print(f"\nBest Cross-Validated ROC-AUC: {search.best_score_:.4f}")
print("Best Hyperparameters:", search.best_params_)

# Final test on the most recent data (the last fold)
train_idx, test_idx = list(tscv.split(X))[-1]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)

print(f"\nFinal Test Split Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Final Test Split ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")


Starting Hyperparameter Tuning...
Fitting 5 folds for each of 15 candidates, totalling 75 fits


c:\Users\rohan\OneDrive - Monash University\Coding\Tennis Prediction\atp_scraper\Lib\site-packages\xgboost\training.py:199: UserWarning: [12:17:28] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Best Cross-Validated ROC-AUC: 0.7124
Best Hyperparameters: {'xgb__subsample': 0.9, 'xgb__n_estimators': 300, 'xgb__max_depth': 6, 'xgb__learning_rate': 0.01, 'xgb__gamma': 0, 'xgb__colsample_bytree': 0.8}

Final Test Split Accuracy: 0.6832
Final Test Split ROC-AUC: 0.7630


In [26]:
model_filename = 'tennis_prediction_pipeline.joblib'

# This saves everything: the XGBoost model AND the pipeline structure
joblib.dump(best_model, model_filename)

print(f"Pipeline saved as {model_filename}")

Pipeline saved as tennis_prediction_pipeline.joblib


--- Prediction: Jannik Sinner vs Casper Ruud ---
Surface: Clay | Date: 2025-09-14
Predicted Winner: Jannik Sinner
Confidence: 92.05%
Win Probability for Jannik Sinner: 92.05%
Win Probability for Casper Ruud: 7.95%
